In [0]:
# =============================================================================
# sql_vs_pipeline_diff  -  READ ONLY. Confirm the SQL->PySpark TRANSLATION is faithful.
# The 2311/2307 changes are authored as SQL-Server; the live segmentation is PySpark in
# SILVER_ACTIVE_APPEALS. This diffs, per case, the SQL v5 result vs the pipeline's output:
#   SQL v5  -> CaseNo, TargetState   (dev runs v5 on the obfuscated DB, exports, loads it here)
#   PIPELINE-> stg_segmentation_states  (CaseNo, TargetState)
# Reports MATCH / MISMATCH (state A in SQL, state B in pipeline) / SQL-only / pipeline-only.
# Any non-match = a translation bug to fix. Single print.
# =============================================================================

In [0]:
# ---- CELL 0 : config + auth ----
# SQL v5 result: set ONE of these (a table dev loaded, or a CSV/parquet path they exported).
SQL_RESULT_TBL  = ""     # e.g. "hive_metastore.test_reporting.seg_sql_v5"   (CaseNo + TargetState)
SQL_RESULT_PATH = ""     # e.g. "dbfs:/tmp/seg_sql_v5.csv"  (CSV with header CaseNo,TargetState) or .parquet
PIPELINE_TBL    = "hive_metastore.ariadm_active_appeals.stg_segmentation_states"

from pyspark.sql import functions as F
from pyspark.sql.functions import *
REPORT=[]
def log(*a): REPORT.append(" ".join(str(x) for x in a))
def logdf(df,n=40):
    try: REPORT.append(df._jdf.showString(n,0,False))
    except Exception as e: REPORT.append(f"  (render fail: {str(e)[:100]})")
_c=spark.read.option("multiline","true").json("dbfs:/configs/config.json")
env_name=_c.first()["env"].strip().lower(); lz_key=_c.first()["lz_key"].strip().lower()
KV=f"ingest{lz_key}-meta002-{env_name}"
cid=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-CLIENT-ID"); csec=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-CLIENT-SECRET"); tid=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-TENANT-ID")
for sa in [f"ingest{lz_key}curated{env_name}",f"ingest{lz_key}raw{env_name}"]:
    spark.conf.set(f"fs.azure.account.auth.type.{sa}.dfs.core.windows.net","OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{sa}.dfs.core.windows.net","org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{sa}.dfs.core.windows.net",cid)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{sa}.dfs.core.windows.net",csec)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{sa}.dfs.core.windows.net",f"https://login.microsoftonline.com/{tid}/oauth2/token")
def col_ci(cols,name): return next((c for c in cols if c.lower()==name.lower()),None)
def norm(df):
    cc=col_ci(df.columns,"CaseNo"); tc=col_ci(df.columns,"TargetState") or col_ci(df.columns,"Segment") or col_ci(df.columns,"State")
    return df.select(trim(col(cc)).alias("CaseNo"), col(tc).alias("state")).distinct()

In [0]:
# ---- CELL 1 : load both sides ----
log("="*74); log("SQL v5  vs  PIPELINE segmentation - per-case translation check"); log("="*74)
if SQL_RESULT_TBL:
    sql=norm(spark.table(SQL_RESULT_TBL)); log(f"SQL side: table {SQL_RESULT_TBL}")
elif SQL_RESULT_PATH:
    rd=spark.read.option("header","true").csv(SQL_RESULT_PATH) if SQL_RESULT_PATH.endswith(".csv") else spark.read.parquet(SQL_RESULT_PATH)
    sql=norm(rd); log(f"SQL side: file {SQL_RESULT_PATH}")
else:
    log("!! Set SQL_RESULT_TBL or SQL_RESULT_PATH to the v5 SQL output (CaseNo+TargetState). Nothing to compare.")
    print("\n".join(REPORT)); dbutils.notebook.exit("no sql result")
pipe=norm(spark.table(PIPELINE_TBL)); log(f"PIPELINE: {PIPELINE_TBL}")
log(f"SQL cases={sql.count()}   pipeline cases={pipe.count()}")

In [0]:
# ---- CELL 2 : diff ----
j=(sql.withColumnRenamed("state","sql_state")
      .join(pipe.withColumnRenamed("state","pipe_state"), "CaseNo", "full_outer"))
match     = j.filter(col("sql_state").eqNullSafe(col("pipe_state")) & col("sql_state").isNotNull())
mismatch  = j.filter(col("sql_state").isNotNull() & col("pipe_state").isNotNull() & ~col("sql_state").eqNullSafe(col("pipe_state")))
sql_only  = j.filter(col("pipe_state").isNull())
pipe_only = j.filter(col("sql_state").isNull())
nM,nX,nS,nP = match.count(),mismatch.count(),sql_only.count(),pipe_only.count()
log("\n"+"-"*74)
log(f"MATCH (same state both sides):     {nM}")
log(f"MISMATCH (different state):        {nX}   <-- translation bug")
log(f"SQL-only (in SQL, not pipeline):   {nS}   <-- pipeline missing these")
log(f"PIPELINE-only (not in SQL):        {nP}   <-- pipeline has extra")
if nX:
    log("\n>>> MISMATCH cross-tab (sql_state -> pipe_state):")
    logdf(mismatch.groupBy("sql_state","pipe_state").count().orderBy(desc("count")),60)
    log("\nsample mismatches:"); logdf(mismatch.select("CaseNo","sql_state","pipe_state").limit(40))
if nS: log("\nsample SQL-only:"); logdf(sql_only.select("CaseNo","sql_state").limit(20))
if nP: log("\nsample PIPELINE-only:"); logdf(pipe_only.select("CaseNo","pipe_state").limit(20))

In [0]:
# ---- CELL 3 : verdict + single print ----
ok = (nX==0 and nS==0 and nP==0)
log("\n"+"="*74)
log(f">>> VERDICT: {'PASS - pipeline is a faithful translation of the SQL (every case agrees)' if ok else 'FAIL - '+str(nX+nS+nP)+' case(s) differ (see mismatch cross-tab / only-lists)'}")
log("READ: MISMATCH cross-tab pinpoints WHICH state pair is wrong -> maps to the .when() block dev mis-translated.")
log("      NOTE: SQL uses the retention ref date of the run (staging=cut date '2026-06-05'); ensure the pipeline")
log("      that built stg_segmentation_states used the SAME reference, else high-level CCD/archive counts differ.")
full="\n".join(REPORT)
try:
    from datetime import datetime
    user=spark.sql("SELECT current_user()").first()[0]; ts=datetime.now().strftime("%Y%m%d_%H%M%S")
    folder=f"/Workspace/Users/{user}/Results/sql_vs_pipeline/{ts}"; dbutils.fs.mkdirs(f"file:{folder}")
    p=f"{folder}/sql_vs_pipeline_diff.txt"; open(p,"w").write(full); full+=f"\n\n>>> saved to: {p}"
except Exception as e: full+=f"\n(save failed: {str(e)[:80]})"
print(full)